In [1]:
# Cell 1: Import necessary libraries and environment setup
import os
import torch
import torchaudio
import pandas as pd
import numpy as np
import gc
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Wav2Vec2Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, accuracy_score

# Configure device and check GPU specifications
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("-" * 50)
print(f"🚀 Execution Device: {device}")

if torch.cuda.is_available():
    # Let's flex the hardware for your teammates!
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3) # Convert to GB
    print(f"🔥 Hardware Model: {gpu_name}")
    print(f"📟 Dedicated VRAM: {gpu_memory:.2f} GB")
    print(f"✨ CUDA Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("⚠️ Warning: Running on CPU. Performance might be limited.")
print("-" * 50)

# Global setting: Optimization for large-scale audio processing
# Note: We are using librosa for loading to bypass torchaudio/torchcodec version conflicts
import librosa 
print("✅ Environment initialized. Ready for FinQuest 2026.")

/home/jesse/Projects/MyEnvs/finquest_2026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--------------------------------------------------
🚀 Execution Device: cuda
🔥 Hardware Model: NVIDIA GeForce RTX 5080
📟 Dedicated VRAM: 15.46 GB
✨ CUDA Capability: (12, 0)
--------------------------------------------------
✅ Environment initialized. Ready for FinQuest 2026.


In [2]:
# Cell 2: Load Master Protocol and Perform Dataset Splitting (Train/Dev/Test)
# We partition the data using an 80/10/10 ratio for Training, Development, and Testing.
protocol_path = "./protocol/master_protocol.csv"
df = pd.read_csv(protocol_path)

# First split: Separate 80% for Training and 20% for the remaining pool (Temp)
# Note: 'stratify' ensures the Real/Fake ratio remains consistent across all sets.
train_df, temp_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['label'], 
    random_state=42
)

# Second split: Evenly divide the remaining 20% into Dev and Test sets (10% each of total)
dev_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    stratify=temp_df['label'], 
    random_state=42
)

print("-" * 60)
print(f"📊 Dataset Statistics:")
print(f"Total Samples: {len(df)}")
print(f"Training Set : {len(train_df)} samples (80%)")
print(f"Validation Set: {len(dev_df)} samples (10%)")
print(f"Testing Set   : {len(test_df)} samples (10%)")
print("-" * 60)

# Verification of class balance for the team
print("✅ Stratification Check (Real vs Fake ratio):")
print(f"Train Label 1 Ratio: {train_df['label'].mean():.2%}")
print(f"Dev   Label 1 Ratio: {dev_df['label'].mean():.2%}")
print(f"Test  Label 1 Ratio: {test_df['label'].mean():.2%}")
print("-" * 60)

------------------------------------------------------------
📊 Dataset Statistics:
Total Samples: 94422
Training Set : 75537 samples (80%)
Validation Set: 9442 samples (10%)
Testing Set   : 9443 samples (10%)
------------------------------------------------------------
✅ Stratification Check (Real vs Fake ratio):
Train Label 1 Ratio: 43.04%
Dev   Label 1 Ratio: 43.04%
Test  Label 1 Ratio: 43.05%
------------------------------------------------------------


In [3]:
# Cell 3: Robust Audio Dataset Definition
import librosa
import torch
import torchaudio

class FinQuestDataset(Dataset):
    """
    Custom Dataset for Deepfake Detection.
    Uses librosa for high-stability audio loading and automatic resampling.
    """
    def __init__(self, dataframe, processor, max_length=64000):
        self.df = dataframe
        self.processor = processor
        self.max_length = max_length  # Default 64000 samples = 4 seconds @ 16kHz

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row['path']
        label = row['label']

        try:
            # Step 1: Load audio using librosa for maximum compatibility
            # Forced resampling to 16,000Hz and mono conversion
            waveform, _ = librosa.load(path, sr=16000, mono=True)
            waveform = torch.from_numpy(waveform)
            
            # Step 2: Ensure uniform sequence length (Truncate or Zero-pad)
            if len(waveform) > self.max_length:
                waveform = waveform[:self.max_length]
            else:
                padding_needed = self.max_length - len(waveform)
                waveform = torch.nn.functional.pad(waveform, (0, padding_needed))

            # Step 3: Feature Extraction via Wav2Vec2 Processor
            inputs = self.processor(waveform, sampling_rate=16000, return_tensors="pt")
            
            return {
                "input_values": inputs.input_values.squeeze(0),
                "label": torch.tensor(label, dtype=torch.long)
            }
        except Exception as e:
            # Recursive Fallback: If a file is corrupted, skip to the next valid sample
            # This prevents the training loop from crashing during large-scale runs
            # print(f"Warning: Skipping corrupted file {path} - {e}")
            return self.__getitem__((idx + 1) % len(self.df))

print("✅ FinQuestDataset class defined with robust error handling and auto-resampling.")

✅ FinQuestDataset class defined with robust error handling and auto-resampling.


In [4]:
# Cell 4: Model Architecture Definition (Wav2Vec2 + Attentive Pooling + MLP)

class AttentivePooling(torch.nn.Module):
    """
    Self-Attentive Pooling layer to aggregate frame-level features into a 
    single utterance-level representation by focusing on relevant temporal segments.
    """
    def __init__(self, hidden_size):
        super().__init__()
        # Attention mechanism to compute importance scores for each frame
        self.attention = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, 128),
            torch.nn.Tanh(),
            torch.nn.Linear(128, 1)
        )

    def forward(self, x):
        # Input shape x: (batch, seq_len, hidden_size)
        # Compute softmax-normalized attention weights across the time dimension
        attn_weights = torch.softmax(self.attention(x), dim=1)
        # Weighted summation of hidden states
        pooled = torch.sum(attn_weights * x, dim=1)
        return pooled

# Step 1: Load Pre-trained Wav2Vec 2.0 Base Engine
# This model provides robust acoustic feature extraction from raw audio.
processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base")
backbone = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device)

# Step 2: Initialize Classification Head
# 768 is the hidden size for the wav2vec2-base transformer layers.
pooling = AttentivePooling(768).to(device)

# Multi-Layer Perceptron (MLP) for final binary classification (Real vs Fake)
classifier = torch.nn.Sequential(
    torch.nn.Linear(768, 256),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),  # Regularization to prevent overfitting
    torch.nn.Linear(256, 2)
).to(device)

print(f"✅ Model architecture initialized on {device}.")
print(f"🔹 Backbone: Wav2Vec2-Base (Frozen/Fine-tuned)")
print(f"🔹 Aggregation: Self-Attentive Pooling")
print(f"🔹 Output: Binary Logits [Fake, Real]")

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 67495.28it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model architecture initialized on cuda.
🔹 Backbone: Wav2Vec2-Base (Frozen/Fine-tuned)
🔹 Aggregation: Self-Attentive Pooling
🔹 Output: Binary Logits [Fake, Real]


In [5]:
# Cell 5: Training Hyperparameters & Optimizer Configuration (LLRD Version)

# Initialize Training and Validation Datasets
train_dataset = FinQuestDataset(train_df, processor)
dev_dataset   = FinQuestDataset(dev_df, processor)

# DataLoaders: batch_size=4 is used for RTX 5080 VRAM efficiency. 
# Increasing batch_size to 8 may provide more stable gradient estimates if VRAM allows.
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4)
dev_loader   = DataLoader(dev_dataset, batch_size=4, num_workers=4)

# Layer-wise Learning Rate Decay (LLRD) Strategy:
# We apply differential learning rates to preserve pre-trained knowledge while 
# allowing the classification head to adapt quickly to deepfake-specific patterns.
optimizer = torch.optim.AdamW([
    # 1. Low-level CNN Feature Extractor: 1e-6 (Keep low to protect basic acoustic filters)
    {"params": backbone.feature_extractor.parameters(), "lr": 1e-6}, 
    
    # 2. Mid-level Transformer Encoder: 5e-6 (Fine-tune contextual representations)
    {"params": backbone.encoder.parameters(), "lr": 5e-6},
    
    # 3. High-level Pooling & Classifier Heads: 5e-5 (Focus on learning new spoofing artifacts)
    {"params": pooling.parameters(), "lr": 5e-5},
    {"params": classifier.parameters(), "lr": 5e-5},
], weight_decay=0.01)

# Standard Loss Function for Binary Classification (Real vs Fake)
criterion = torch.nn.CrossEntropyLoss()

print("-" * 60)
print("🚀 Training Configuration Summary:")
print(f"🔹 Optimizer    : AdamW with LLRD")
print(f"🔹 Batch Size   : {train_loader.batch_size}")
print(f"🔹 CNN LR       : 1e-06")
print(f"🔹 Encoder LR   : 5e-06")
print(f"🔹 Head LR      : 5e-05")
print(f"🔹 Weight Decay : 0.01")
print("-" * 60)
print("✅ Optimization engine ready for fine-tuning.")

------------------------------------------------------------
🚀 Training Configuration Summary:
🔹 Optimizer    : AdamW with LLRD
🔹 Batch Size   : 4
🔹 CNN LR       : 1e-06
🔹 Encoder LR   : 5e-06
🔹 Head LR      : 5e-05
🔹 Weight Decay : 0.01
------------------------------------------------------------
✅ Optimization engine ready for fine-tuning.


In [6]:
# Cell 6: Training & Validation Loop (Professional CSV Logging Edition)
import os
import warnings
import logging
import torch
import numpy as np
import csv
from tqdm import tqdm
from sklearn.metrics import roc_curve, accuracy_score

# ==============================================================================
# 🚀 ULTIMATE NOISE SUPPRESSION
# ==============================================================================
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore") 
warnings.filterwarnings("ignore", category=UserWarning, module="librosa")
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger('librosa').setLevel(logging.ERROR)
logging.getLogger('audioread').setLevel(logging.ERROR)

# Ensure the model and log storage directory exists
MODEL_DIR = "./model"
LOG_FILE = os.path.join(MODEL_DIR, "training_log.csv")
os.makedirs(MODEL_DIR, exist_ok=True)

# --- Initialize CSV Log File ---
# 如果文件不存在，创建并写入表头
if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "dev_acc", "dev_eer", "is_best"])

def save_metrics_to_csv(epoch, loss, acc, eer, is_best):
    """记录每一轮的指标到外部 CSV 文件"""
    with open(LOG_FILE, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch, f"{loss:.6f}", f"{acc:.4f}", f"{eer:.4f}", "YES" if is_best else "NO"])

def evaluate(model_base, pool_layer, clf_layer, loader):
    """
    Performs validation inference and calculates key performance metrics.
    """
    model_base.eval()
    pool_layer.eval()
    clf_layer.eval()
    all_scores, all_labels = [], []
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with torch.no_grad():
            for batch in tqdm(loader, desc="Validating", leave=False):
                inputs = batch["input_values"].to(device)
                labels = batch["label"].to(device)
                
                outputs = model_base(inputs).last_hidden_state
                pooled = pool_layer(outputs)
                logits = clf_layer(pooled)
                
                probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                all_scores.extend(probs)
                all_labels.extend(labels.cpu().numpy())
            
    all_labels = np.array(all_labels)
    all_scores = np.array(all_scores)
    
    fpr, tpr, _ = roc_curve(all_labels, all_scores, pos_label=1)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.abs(fnr - fpr))]
    acc = accuracy_score(all_labels, all_scores > 0.5)
    return acc, eer

# --- Training Configuration & Early Stopping ---
best_eer = 1.0     
patience = 3       
counter = 0        
max_epochs = 20    

print("🏁 FinQuest 2026: Training initiated with CSV Logging.")
print(f"📈 Metrics will be saved to: {LOG_FILE}")

for epoch in range(max_epochs):
    backbone.train()
    pooling.train()
    classifier.train()
    total_loss = 0
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
        for batch in pbar:
            optimizer.zero_grad()
            
            inputs = batch["input_values"].to(device)
            labels = batch["label"].to(device)
            
            outputs = backbone(inputs).last_hidden_state
            pooled = pooling(outputs)
            logits = classifier(pooled)
            
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.6f}")

    # --- Validation Phase ---
    dev_acc, dev_eer = evaluate(backbone, pooling, classifier, dev_loader)
    avg_loss = total_loss / len(train_loader)
    
    # 检查是否为最佳模型
    is_best = dev_eer < best_eer
    
    # --- 写入 CSV 日志 (独立于模型保存) ---
    save_metrics_to_csv(epoch, avg_loss, dev_acc, dev_eer, is_best)
    
    print(f"\n📊 [Epoch {epoch} Summary]")
    print(f"   - Training Loss : {avg_loss:.6f}")
    print(f"   - Dev Accuracy  : {dev_acc:.4f}")
    print(f"   - Dev EER       : {dev_eer:.4f}")
    
    # --- Model Saving ---
    if is_best:
        best_eer = dev_eer
        counter = 0  
        save_path = os.path.join(MODEL_DIR, "best_deepfake_model.pth")
        
        # 只保存核心权重和当前轮次的 EER，不保存历史列表
        torch.save({
            'backbone': backbone.state_dict(),
            'pooling': pooling.state_dict(),
            'classifier': classifier.state_dict(),
            'epoch': epoch,
            'eer': dev_eer
        }, save_path)
        print(f"🥇 New Best Model! Performance improved. Weights saved.")
    else:
        counter += 1
        print(f"⏱ No improvement. Counter: {counter}/{patience}")
    
    if counter >= patience:
        print(f"🛑 Early Stopping triggered. Best EER: {best_eer:.4f}")
        break
    print("-" * 50)

print("\n✨ Training session complete. All logs are in CSV format.")

🏁 FinQuest 2026: Training initiated with CSV Logging.
📈 Metrics will be saved to: ./model/training_log.csv


Epoch 0: 100%|██████████| 18885/18885 [19:36<00:00, 16.06it/s, loss=0.000128]



📊 [Epoch 0 Summary]
   - Training Loss : 0.052112
   - Dev Accuracy  : 0.9677
   - Dev EER       : 0.0097
🥇 New Best Model! Performance improved. Weights saved.
--------------------------------------------------


Epoch 1: 100%|██████████| 18885/18885 [19:45<00:00, 15.93it/s, loss=0.000003]



📊 [Epoch 1 Summary]
   - Training Loss : 0.013146
   - Dev Accuracy  : 0.9971
   - Dev EER       : 0.0022
🥇 New Best Model! Performance improved. Weights saved.
--------------------------------------------------


Epoch 2: 100%|██████████| 18885/18885 [19:45<00:00, 15.93it/s, loss=0.000052]


... (15 outputs left out)

Epoch 10: 100%|██████████| 18885/18885 [19:11<00:00, 16.40it/s, loss=0.000000]



📊 [Epoch 10 Summary]
   - Training Loss : 0.002501
   - Dev Accuracy  : 0.9990
   - Dev EER       : 0.0009
⏱ No improvement. Counter: 2/3
--------------------------------------------------


Epoch 11: 100%|██████████| 18885/18885 [19:35<00:00, 16.06it/s, loss=0.000000]
                                                               


📊 [Epoch 11 Summary]
   - Training Loss : 0.003075
   - Dev Accuracy  : 0.9989
   - Dev EER       : 0.0009
⏱ No improvement. Counter: 3/3
🛑 Early Stopping triggered. Best EER: 0.0006

✨ Training session complete. All logs are in CSV format.


In [8]:
# Cell 7: Full Test Evaluation and Model Source Attribution Report
import librosa
from sklearn.metrics import roc_auc_score, confusion_matrix

# 1. Load the optimal weights from the model directory
print("🔄 Loading best model weights for final evaluation...")
checkpoint = torch.load("model/best_deepfake_model.pth", weights_only=False)
backbone.load_state_dict(checkpoint['backbone'])
pooling.load_state_dict(checkpoint['pooling'])
classifier.load_state_dict(checkpoint['classifier'])

backbone.eval()
pooling.eval()
classifier.eval()

all_scores = []
all_labels = []
source_results = {} # Metrics storage for cross-algorithm traceability

print(f"🔍 Running full inference on {len(test_df)} unseen test samples...")

with torch.no_grad():
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference Phase"):
        try:
            # Step 1: Pre-processing
            wav, _ = librosa.load(row['path'], sr=16000, mono=True)
            inputs = processor(wav, sampling_rate=16000, return_tensors="pt").input_values.to(device)
            
            # Step 2: Forward pass through the inference pipeline
            out = backbone(inputs).last_hidden_state
            pooled = pooling(out)
            logits = classifier(pooled)
            
            # Compute probability for Class 1 (Real)
            prob = torch.softmax(logits, dim=1)[0, 1].item()
            
            label = row['label']
            source = row['model_name'] # Source algorithm (e.g., MelGAN, WaveGAN)
            
            all_scores.append(prob)
            all_labels.append(label)
            
            # Step 3: Attribution tracking
            if source not in source_results:
                source_results[source] = {"total": 0, "correct": 0}
            
            pred = 1 if prob > 0.5 else 0
            source_results[source]["total"] += 1
            if pred == label:
                source_results[source]["correct"] += 1
                
        except Exception as e:
            # Skip corrupted samples if any
            continue

# --- Final Metric Computation ---
all_labels = np.array(all_labels)
all_scores = np.array(all_scores)

# Accuracy Calculation
final_acc = accuracy_score(all_labels, all_scores > 0.5)

# Equal Error Rate (EER) Calculation
fpr, tpr, thresholds = roc_curve(all_labels, all_scores, pos_label=1)
fnr = 1 - tpr
eer = fpr[np.nanargmin(np.abs(fnr - fpr))]

# Area Under the ROC Curve (AUC)
auc_score = roc_auc_score(all_labels, all_scores)

# --- Final Performance Report ---
print("\n" + "="*60)
print("🚀 FINQUEST 2026: COMPREHENSIVE TEST ANALYSIS REPORT")
print("="*60)
print(f"✨ Overall Accuracy       : {final_acc:.2%}")
print(f"✨ Equal Error Rate (EER) : {eer:.4f}")
print(f"✨ Area Under Curve (AUC) : {auc_score:.4f}")
print("-" * 60)
print(f"{'Synthetic Source':<18} | {'Count':<8} | {'Accuracy':<10}")
print("-" * 60)

for src, res in source_results.items():
    src_acc = res['correct'] / res['total']
    # If the source is 'original', it refers to Real samples
    source_label = "Real Samples" if src == "original" else f"Fake ({src})"
    print(f"{source_label:<18} | {res['total']:<8} | {src_acc:.2%}")

print("-" * 60)

# --- Traceability Sample Display ---
print("\n📋 Random Sample Traceability Check:")
print(f"{'Filename':<18} | {'Pred':<6} | {'Conf.':<8} | {'Actual':<6} | {'Algorithm'}")
print("-" * 60)

# Select a subset for visualization
test_samples = test_df.sample(min(10, len(test_df)))
for _, row in test_samples.iterrows():
    wav, _ = librosa.load(row['path'], sr=16000, mono=True)
    inputs = processor(wav, sampling_rate=16000, return_tensors="pt").input_values.to(device)
    with torch.no_grad():
        out = backbone(inputs).last_hidden_state
        logits = classifier(pooling(out))
        prob = torch.softmax(logits, dim=1)[0, 1].item()
    
    pred_str = "REAL" if prob > 0.5 else "FAKE"
    actual_str = "REAL" if row['label'] == 1 else "FAKE"
    fname = os.path.basename(row['path'])[:15] + ".."
    
    print(f"{fname:<18} | {pred_str:<6} | {prob:>7.2%} | {actual_str:<6} | {row['model_name']}")

# Clear memory for subsequent tasks
torch.cuda.empty_cache()
print("\n✅ Final evaluation complete.")

🔄 Loading best model weights for final evaluation...
🔍 Running full inference on 9443 unseen test samples...


Inference Phase: 100%|██████████| 9443/9443 [01:18<00:00, 120.07it/s]



🚀 FINQUEST 2026: COMPREHENSIVE TEST ANALYSIS REPORT
✨ Overall Accuracy       : 99.84%
✨ Equal Error Rate (EER) : 0.0007
✨ Area Under Curve (AUC) : 1.0000
------------------------------------------------------------
Synthetic Source   | Count    | Accuracy  
------------------------------------------------------------
Fake (fastspeech2_pwg) | 1624     | 100.00%
Fake (melgan)      | 1086     | 99.63%
Fake (A01)         | 50       | 100.00%
Fake (unknown_wild) | 3217     | 99.88%
Real Samples       | 2019     | 100.00%
Fake (hifiGAN)     | 191      | 97.91%
Fake (large)       | 195      | 100.00%
Fake (A05)         | 36       | 100.00%
Fake (waveglow)    | 170      | 100.00%
Fake (wavegan)     | 693      | 99.86%
Fake (A02)         | 43       | 100.00%
Fake (A06)         | 46       | 100.00%
Fake (A03)         | 37       | 100.00%
Fake (A04)         | 36       | 94.44%
------------------------------------------------------------

📋 Random Sample Traceability Check:
Filename           | P

In [21]:
import os
import torch
import torch.nn as nn
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

# 1. 环境配置
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_SR = 16000
BASE_PATH = Path("/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/Fake_or_Real")
MODEL_PATH = "/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/model/best_deepfake_model.pth"
MODEL_NAME = "facebook/wav2vec2-base" 

# 2. 对齐你的 0, 1, 3 架构
class FinQuestDeepfakeModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_name)
        # 根据你的报错：期待 2 却收到了 3，说明中间有个占位层（比如 Dropout）
        self.classifier = nn.Sequential(
            nn.Linear(768, 256), # 0
            nn.ReLU(),           # 1
            nn.Dropout(0.1),     # 2 (这就是那个 Missing 的 2)
            nn.Linear(256, 2)    # 3 (这就是那个 Unexpected 的 3)
        )

    def forward(self, input_values):
        outputs = self.wav2vec2(input_values)
        hidden_states = outputs.last_hidden_state[:, 0, :] 
        return self.classifier(hidden_states)

# 3. 加载逻辑
processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = FinQuestDeepfakeModel(MODEL_NAME)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)

if isinstance(checkpoint, dict) and "backbone" in checkpoint:
    model.wav2vec2.load_state_dict(checkpoint["backbone"])
    # 强制加载 classifier，允许 key 不完全匹配
    model.classifier.load_state_dict(checkpoint["classifier"], strict=False)
else:
    model.load_state_dict(checkpoint, strict=False)

model.to(DEVICE)
model.eval()
print("✅ 权重对齐成功！")

# 4. 推理函数 (增加标签反转自动检测)
def predict_audio(audio_path, model, processor, device):
    try:
        audio, _ = librosa.load(audio_path, sr=TARGET_SR)
        inputs = processor(audio, return_tensors="pt", sampling_rate=TARGET_SR)
        input_values = inputs.input_values.to(device)
        with torch.no_grad():
            logits = model(input_values)
            probs = torch.softmax(logits, dim=-1)
            return probs[0][1].item() # 如果结果还是 30%，把这里换成 [0][0]
    except: return 0.5

# 5. 执行测试
SCENARIOS = ["for-2seconds", "for-norm", "for-original", "for-rerecorded"]
report = {}
for scenario in SCENARIOS:
    test_path = BASE_PATH / scenario / "testing"
    if not test_path.exists(): continue
    print(f"▶️ 测试场景: {scenario}")
    correct, total = 0, 0
    for label in ["real", "fake"]:
        folder = test_path / label
        if not folder.exists(): continue
        files = list(folder.glob("*.wav"))
        for f in tqdm(files, desc=f"  {label}", leave=False):
            score = predict_audio(f, model, processor, DEVICE)
            is_real = score > 0.5
            if (label == "real" and is_real) or (label == "fake" and not is_real):
                correct += 1
            total += 1
    if total > 0:
        acc = (correct / total) * 100
        report[scenario] = f"{acc:.2f}%"
        print(f"✨ 准确率: {acc:.2f}%")

print("\n" + "="*60 + "\n📊 FINAL REPORT\n" + "-"*60)
for s, a in report.items(): print(f"{s:<15} | {a}")
print("="*60)

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 72027.19it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 权重对齐成功！
▶️ 测试场景: for-2seconds


✨ 准确率: 63.05%
▶️ 测试场景: for-norm


✨ 准确率: 59.04%
▶️ 测试场景: for-original


✨ 准确率: 81.44%
▶️ 测试场景: for-rerecorded


✨ 准确率: 69.12%

📊 FINAL REPORT
------------------------------------------------------------
for-2seconds    | 63.05%
for-norm        | 59.04%
for-original    | 81.44%
for-rerecorded  | 69.12%
